In [1]:
!pip install transformers torch tqdm

In [2]:
import os
os.listdir("/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv")

['nasdaq_exteral_data.csv']

In [3]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from tqdm import tqdm

# Load FinBERT
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# File path
file_path = "/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv/nasdaq_exteral_data.csv"

chunksize = 20000
batch_size = 32

output_file = "/kaggle/working/sentiment_output.csv"

# Count rows for progress estimation
total_rows = sum(1 for _ in open(file_path)) - 1
total_chunks = total_rows // chunksize + 1

print("Total rows:", total_rows)
print("Total chunks:", total_chunks)

# Process dataset
for chunk in tqdm(pd.read_csv(file_path, chunksize=chunksize),
                  total=total_chunks,
                  desc="Processing CSV Chunks"):

    chunk = chunk[[
        "Date",
        "Stock_symbol",
        "Article_title",
        "Article"
    ]]

    texts = (chunk["Article_title"].fillna("") + " " +
             chunk["Article"].fillna("")).tolist()

    pos_list = []
    neg_list = []
    neu_list = []

    for i in tqdm(range(0, len(texts), batch_size),
                  desc="FinBERT batches",
                  leave=False):

        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

        pos_list.extend(probs[:,0])
        neg_list.extend(probs[:,1])
        neu_list.extend(probs[:,2])

    chunk["pos"] = pos_list
    chunk["neg"] = neg_list
    chunk["neu"] = neu_list

    chunk[["Date","Stock_symbol","pos","neg","neu"]].to_csv(
        output_file,
        mode="a",
        header=not pd.io.common.file_exists(output_file),
        index=False
    )

    torch.cuda.empty_cache()

print("Sentiment extraction completed.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
sentiment = pd.concat(sentiment_rows)

sentiment["Date"] = pd.to_datetime(sentiment["Date"])

sentiment.to_csv("sentiment_output.csv", index=False)

In [ ]:
daily_sentiment = sentiment.groupby(
    ["Stock_symbol","Date"]
)[["pos","neg","neu"]].mean().reset_index()